In [ ]:
# System Check
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU name: Tesla T4


In [ ]:
# Install Libraries
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
# Mount Drive (If using Colab)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load Prepared Data
import pandas as pd
from datasets import load_dataset

# ---- EDIT PATHS TO WHERE YOU SAVED PART 1 FILES ----
TRAIN_PATH = "/content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/t5_hyde_train_split.jsonl"
VAL_PATH   = "/content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/t5_hyde_val_split.jsonl"
# ----------------------------------------------------

# Load into Hugging Face Dataset format
dataset = load_dataset("json", data_files={"train": TRAIN_PATH, "validation": VAL_PATH})
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 10800
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 1200
    })
})


In [ ]:
# Load Model and Tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# MODEL_NAME = "google/flan-t5-small"
MODEL_NAME = "razent/SciFive-base-Pubmed_PMC"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model and Tokenizer loaded successfully!")

Loading razent/SciFive-base-Pubmed_PMC...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Model and Tokenizer loaded successfully!


In [ ]:
# Pre-processing (Tokenization)
# CRITICAL CHANGE: Increased target_length to 256 for paragraph generation
def tokenize_function(batch):
    # Inputs (Questions) are short
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=128,
        truncation=True
    )

    # Targets (Hypothetical Answers) are longer paragraphs
    labels = tokenizer(
        batch["target_text"],
        max_length=256,  # <--- Increased from 64 to 256
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)

print("Data tokenized.")

Map:   0%|          | 0/10800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Data tokenized.


In [ ]:
# Training Setup
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./hyde_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,              # Slightly higher LR for generation usually helps T5
    per_device_train_batch_size=8,   # Lower batch size (8) because targets are longer (256 tokens)
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

/tmp/ipython-input-3694315290.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
# Train
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss
1,1.968000,1.811950
2,1.793000,1.777777
3,1.677800,1.774317


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=4050, training_loss=1.8206560149016204, metrics={'train_runtime': 2025.2869, 'train_samples_per_second': 15.998, 'train_steps_per_second': 2.0, 'total_flos': 1714796874915840.0, 'train_loss': 1.8206560149016204, 'epoch': 3.0})

In [ ]:
# Test the Model (Hypothetical Answer Generation)
def generate_hypothetical_answer(question):
    input_text = "hypothetical answer: " + question
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=256,       # Allow it to generate a long paragraph
            num_beams=4,          # Beam search for better quality
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_q = "What are the different measurements and calculations performed during echocardiography?"
print(f"\nQuestion: {test_q}")
print(f"Generated HyDE Document:\n{generate_hypothetical_answer(test_q)}")


Question: What are the different measurements and calculations performed during echocardiography?
Generated HyDE Document:



In [ ]:
# Save Model
# SAVE_DIR = "/content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/flan_t5_hyde_generator"
SAVE_DIR = "/content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to: {SAVE_DIR}")

Model saved to: /content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator


In [ ]:
# Zip and Download
# !zip -r flan_t5_hyde_generator.zip {SAVE_DIR}
!zip -r scifive_hyde_generator.zip {SAVE_DIR}
from google.colab import files
# files.download("flan_t5_hyde_generator.zip")
files.download("scifive_hyde_generator.zip")

updating: content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator/ (stored 0%)
updating: content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator/config.json (deflated 48%)
updating: content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator/generation_config.json (deflated 27%)
updating: content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator/model.safetensors (deflated 8%)
updating: content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator/tokenizer_config.json (deflated 95%)
updating: content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator/special_tokens_map.json (deflated 86%)
updating: content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator/spiece.model (deflated 48%)
updating: content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator/tokenizer.json (deflated 74%)
updating: co

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator"

In [ ]:
ls /content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/flan_t5_hyde_generator

config.json             special_tokens_map.json  tokenizer.json
generation_config.json  spiece.model             training_args.bin
model.safetensors       tokenizer_config.json


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

repo_name = "hyde-sciFive-cardiology-generator"
username = "SandunR"  # replace this

api.create_repo(
    repo_id=f"{username}/{repo_name}",
    exist_ok=True,
    private=False
)

api.upload_folder(
    folder_path="/content/drive/MyDrive/Research_My_Component/HyDE_Finetuning/scifive_hyde_generator",
    repo_id=f"{username}/{repo_name}",
    repo_type="model"
)

print("✅ Model uploaded successfully!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...de_generator/spiece.model: 100%|##########|  792kB /  792kB            

  ...nerator/model.safetensors:   0%|          |  553kB /  892MB            

  ...nerator/training_args.bin:  20%|#9        | 1.17kB / 5.97kB            

✅ Model uploaded successfully!
